# Index-Aufbau — Entwicklungsnotebook

Dieses Notebook zeigt die Entwicklung des Index-Aufbaus in zwei Phasen:

1. **Fruehes Prototyp** — direkte Nutzung von HuggingFaceBgeEmbeddings und ChromaDB,
   Cosine-Similarity-Demo
2. **Finale Implementierung** — Nutzung von `src.indexing` und `src.hybrid`


---
## Teil 1: Fruehes Prototyp

Erste Experimente vor Entwicklung des `src.indexing`-Moduls.
Dieser Code ist nicht mehr Teil der aktiven Pipeline.


In [ ]:
# Pakete installieren (nur einmalig noetig)
# %pip install langchain langchain-community chromadb
# %pip install sentence-transformers huggingface-hub
# %pip install pypdf


In [ ]:
from pathlib import Path
import os
import sys

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('ROOT:', ROOT)

# BGE-M3 Embedding-Modell laden
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

embedder = HuggingFaceBgeEmbeddings(
    model_name='BAAI/bge-m3',
    encode_kwargs={'normalize_embeddings': True},
)
print('Embedder geladen.')


In [ ]:
# PDFs laden und chunken (Prototyp-Variante)
# Hinweis: read_pdf und chunk_documents existieren nicht mehr.
# Die finale Variante nutzt src.processing (siehe Teil 2).
from src.processing import IngestConfig, load_and_clean_pdf, split_into_chunks

DATA_RAW = ROOT / 'data' / 'raw'
pdf_files = list(DATA_RAW.rglob('*.pdf'))
print(f'{len(pdf_files)} PDFs gefunden:')
for p in pdf_files:
    print(' -', p.name)

all_docs = []
for p in pdf_files:
    docs, _ = load_and_clean_pdf(p, IngestConfig(), verbose=False)
    all_docs.extend(docs)

chunks, _ = split_into_chunks(all_docs, IngestConfig(chunk_size=1200, chunk_overlap=200))
print(f'Seiten: {len(all_docs)} | Chunks: {len(chunks)}')


In [ ]:
# ChromaDB direkt befuellen (Prototyp-Variante)
# Hinweis: In der finalen Pipeline wird build_index() aus src.indexing verwendet.
from langchain_community.vectorstores import Chroma

CHROMA_DIR = ROOT / 'artifacts' / 'chroma_prototype'
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

vs = Chroma.from_documents(
    documents=chunks,
    embedding=embedder,
    persist_directory=str(CHROMA_DIR),
)
print('Vectorstore gespeichert unter:', CHROMA_DIR)


In [ ]:
# Retrieval-Test auf dem Prototyp-Vectorstore
retriever = vs.as_retriever(search_kwargs={'k': 3})
query = 'Was ist der Anwendungsbereich dieser Norm?'
results = retriever.invoke(query)

print('Anfrage:', query)
for i, r in enumerate(results, 1):
    print()
    print(f'--- Treffer {i} ---')
    print('Quelle:', r.metadata.get('source'), '| Seite:', r.metadata.get('page'))
    print(r.page_content[:300], '...')


In [ ]:
# Cosine-Similarity-Demo
# ChromaDB nutzt intern Cosine-Similarity zum Vergleich von Vektoren.
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

vec1 = embedder.embed_query('Bremsen sind Vorrichtungen zum Verzoegern oder Anhalten der Bewegung eines Fahrzeugs.')
vec2 = embedder.embed_query('Der Zug muss rechtzeitig bremsen, um sicher an der Station anzuhalten.')

score = cosine_similarity(vec1, vec2)
print(f'Cosine-Similarity: {score:.4f}')


---
## Teil 2: Finale Implementierung mit `src.indexing`

Nutzung des fertigen `src.indexing`-Moduls zum Aufbau und Laden des Index,
sowie `src.hybrid` fuer Hybrid-Retrieval (BM25 + Dense).


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import sys
from pathlib import Path
import os

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('ROOT:', ROOT)


In [ ]:
from src.indexing import build_index, load_index
from src.config import CHROMA_DIR

# Index neu aufbauen (nur wenn noetig, force_rebuild=True ueberschreibt bestehenden)
# vs = build_index(force_rebuild=True)

# Bestehenden Index laden
vs = load_index()
print('Index geladen aus:', CHROMA_DIR)


In [ ]:
# Hybrid-Retrieval testen (BM25 + Dense via RRF)
from src.hybrid import hybrid_retrieve

query = 'Was ist der Anwendungsbereich dieser Norm?'
docs = hybrid_retrieve(query, k=5, weights=(0.5, 0.5))

print(f'Hybrid-Treffer: {len(docs)}')
for i, d in enumerate(docs, 1):
    print()
    print(f'--- Treffer {i} ---')
    print('Quelle:', d.metadata.get('source'), '| Seite:', d.metadata.get('page'))
    print(d.page_content[:300], '...')
